In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams.update({"figure.dpi": 300})

In [ ]:
# TODO: set the correct path, output of the eval.py script
METRICS_CSV = "/path/to/result.csv"
TM_SCORE_SOLVED_THRESHOLD = 0.45  # According to USalign paper supplementary

In [ ]:
def plot_metric_distribution(
    metrics_csv: str,
    metric: str,
    use_boxplot: bool = True,
    use_swarmplot: bool = True,
    x_axis_label: str = "Predictor",
    y_axis_label: str = "Score",
    ylim: tuple[float, float] | None = None,
    hline: float | None = None,
) -> None:
    data = pd.read_csv(metrics_csv)

    if use_boxplot:
        ax = sns.boxplot(x="predictor", y=metric, data=data)
    
    if use_swarmplot:
        ax = sns.swarmplot(x="predictor", y=metric, data=data, dodge=True, alpha=0.6)

    ax.set_xlabel(x_axis_label)
    ax.set_ylabel(y_axis_label)

    if ylim is not None:
        ax.set(ylim=ylim)
    
    if hline is not None:
        ax.axhline(y=hline, ls="--", color="red")

In [ ]:
# TM-score
plot_metric_distribution(METRICS_CSV, "tm_score", y_axis_label="TM-score", ylim=(0, 1), hline=TM_SCORE_SOLVED_THRESHOLD, use_boxplot=True)

In [ ]:
# RMSD
plot_metric_distribution(METRICS_CSV, "rmsd", y_axis_label="RMSD", use_boxplot=True)

In [ ]:
# INF_ALL
plot_metric_distribution(METRICS_CSV, "inf_all", y_axis_label="INF_ALL", ylim=(0, 1), use_boxplot=True)


In [ ]:
# INF_STACK
plot_metric_distribution(METRICS_CSV, "inf_stack", y_axis_label="INF_STACK", ylim=(0, 1), use_boxplot=True)

In [ ]:
# INF_WC
plot_metric_distribution(METRICS_CSV, "inf_wc", y_axis_label="INF_WC", ylim=(0, 1), use_boxplot=True)

In [ ]:
# INF_NWC
plot_metric_distribution(METRICS_CSV, "inf_nwc", y_axis_label="INF_NWC", ylim=(0, 1), use_boxplot=True)

In [ ]:
# lDDT_C3'
# TODO: might need change if it's not C3 prime based lDDT
plot_metric_distribution(METRICS_CSV, "lddt_c3prime", y_axis_label="lDDT_C3'", ylim=(0, 1), use_boxplot=True)

In [ ]:
def plot_metric_scatter(
    metrics_csv: str,
    predictor_x: str,
    predictor_y: str,
    metric: str,
    label_points: bool = True,
    x_label: str | None = None,
    y_label: str | None = None,
    axis_limits: tuple | None = None,
    title: str | None = None,
    label_offset: float = 0.005,
):
    df = pd.read_csv(metrics_csv)

    # Filter predictors
    df_x = df[df["predictor"] == predictor_x]
    df_y = df[df["predictor"] == predictor_y]

    # Merge on struct_id
    merged = pd.merge(
        df_x[["struct_id", metric]],
        df_y[["struct_id", metric]],
        on="struct_id",
        suffixes=("_x", "_y"),
    )

    if merged.empty:
        raise ValueError("No matching struct_id values found between predictors.")

    # Plot
    plt.figure(figsize=(6, 6))
    plt.scatter(merged[f"{metric}_x"], merged[f"{metric}_y"])

    # Diagonal y = x
    if axis_limits is not None:
        plt.plot(axis_limits, axis_limits, "r--")
        plt.xlim(axis_limits)
        plt.ylim(axis_limits)
    else:
        min_val = min(
            merged[f"{metric}_x"].min(),
            merged[f"{metric}_y"].min()
        )
        max_val = max(
            merged[f"{metric}_x"].max(),
            merged[f"{metric}_y"].max()
        )
        plt.plot([min_val, max_val], [min_val, max_val], "r--")

    # Labels
    if label_points:
        for _, row in merged.iterrows():
            plt.text(
                row[f"{metric}_x"] + label_offset,
                row[f"{metric}_y"] + label_offset,
                row["struct_id"],
                fontsize=9,
            )

    # Axis labels
    plt.xlabel(x_label or f"{predictor_x} {metric}")
    plt.ylabel(y_label or f"{predictor_y} {metric}")

    # Title
    if title:
        plt.title(title)

    plt.gca().set_aspect("equal", adjustable="box")
    plt.tight_layout()
    plt.show()

In [ ]:
PREDICTOR_1 = "predictor_1"  # TODO
PREDICTOR_2 = "predictor_2"  # TODO

plot_metric_scatter(
    METRICS_CSV,
    predictor_x=PREDICTOR_1,
    predictor_y=PREDICTOR_2,
    metric="tm_score",
    label_points=True,
    x_label=f"{PREDICTOR_1} TM-score",
    y_label=f"{PREDICTOR_2} TM-score",
    axis_limits=(0, 1),
    title=f"Comparison of TM-score between {PREDICTOR_1} and {PREDICTOR_2}",
)


In [ ]:
def plot_metric_radar(
    metrics_csv: str,
    predictors: list[str],
    metric: str,
    title: str | None = None,
    reference_value: float | None = None,
):
    """
    Radar plot where:
    - Each axis is a struct_id
    - Each line/color is a predictor
    - Metric values must be in [0, 1]
    - Optional dashed reference circle (e.g., TM-score = 0.45)
    """
    df = pd.read_csv(metrics_csv)

    if reference_value is not None and not (0 <= reference_value <= 1):
        raise ValueError("reference_value must be in [0, 1].")

    # Filter predictors
    df_filt = df[df["predictor"].isin(predictors)]

    pivot = (
        df_filt
        .pivot(index="struct_id", columns="predictor", values=metric)
        .dropna()
    )

    if pivot.empty:
        raise ValueError("No struct_id has all predictors available.")

    if (pivot.values < 0).any() or (pivot.values > 1).any():
        raise ValueError(f"Metric '{metric}' must be in the range [0, 1].")

    struct_ids = pivot.index.tolist()
    num_structs = len(struct_ids)

    angles = np.linspace(0, 2 * np.pi, num_structs, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

    # Plot predictors
    for predictor in predictors:
        values = pivot[predictor].tolist()
        values += values[:1]

        ax.plot(angles, values, linewidth=2, label=predictor)
        ax.fill(angles, values, alpha=0.1)

    # ---- Reference circle ----
    if reference_value is not None:
        ax.plot(
            angles,
            [reference_value] * len(angles),
            "r--",
            linewidth=1.5,
        )

    # ---- Struct ID labels ----
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])

    label_radius = 1.08
    for angle, label in zip(angles[:-1], struct_ids):
        rotation = np.degrees(angle)
        alignment = "left"

        if np.pi / 2 < angle < 3 * np.pi / 2:
            rotation += 180
            alignment = "right"

        ax.text(
            angle,
            label_radius,
            label,
            size=9,
            rotation=rotation,
            rotation_mode="anchor",
            horizontalalignment=alignment,
            verticalalignment="center",
        )

    # ---- Radial axis ----
    ax.set_ylim(0, 1)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.set_yticklabels([f"{v:.1f}" for v in np.linspace(0, 1, 6)])
    ax.tick_params(axis="y", labelsize=8)

    # ---- Title & legend ----
    if title:
        ax.set_title(title, y=1.18)

    ax.legend(loc="upper right", bbox_to_anchor=(1.30, 1.1))

    plt.subplots_adjust(top=0.80)
    plt.show()


In [ ]:
PREDICTOR_1 = "predictor_1"  # TODO
PREDICTOR_2 = "predictor_2"  # TODO

plot_metric_radar(
    METRICS_CSV,
    predictors=[PREDICTOR_1, PREDICTOR_2],
    metric="tm_score",
    title="Radar Plot of TM-score across predictors",
    reference_value=TM_SCORE_SOLVED_THRESHOLD,
)